# 02 — Tune Linear Models

Optuna study comparing LogisticRegression, SVM, and NaiveBayes
on the static feature set. Best model logged to MLflow.

In [ ]:
input_dir = "data/processed"
n_trials = 50
random_state = 42
cv_folds = 5

In [ ]:
import numpy as np
import pandas as pd
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
import mlflow

In [ ]:
# ── Load data ──
X_train = pd.read_parquet(f"{input_dir}/X_train.parquet")
y_train = pd.read_parquet(f"{input_dir}/y_train.parquet")["y"]
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
print(f"Loaded {len(X_train)} training rows, {X_train.shape[1]} features")

In [ ]:
cv = StratifiedKFold(cv_folds, shuffle=True, random_state=random_state)


def objective(trial):
    model_name = trial.suggest_categorical("model", ["lr", "svm", "nb"])

    if model_name == "lr":
        C = trial.suggest_float("C", 1e-4, 1e2, log=True)
        penalty = trial.suggest_categorical("penalty", ["l2", None])
        model = LogisticRegression(
            C=C, penalty=penalty, solver="lbfgs", max_iter=1000, random_state=random_state
        )
    elif model_name == "svm":
        C = trial.suggest_float("C", 1e-4, 1e2, log=True)
        kernel = trial.suggest_categorical("kernel", ["rbf", "linear", "poly"])
        gamma = trial.suggest_float("gamma", 1e-4, 1, log=True) if kernel != "linear" else "scale"
        model = SVC(C=C, kernel=kernel, gamma=gamma, probability=True, random_state=random_state)
    else:
        var_smoothing = trial.suggest_float("var_smoothing", 1e-12, 1e-2, log=True)
        model = GaussianNB(var_smoothing=var_smoothing)

    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring="roc_auc")
    return scores.mean()


study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=n_trials)
print(f"Best trial: {study.best_trial.value:.4f} ROC-AUC")
print(f"Best params: {study.best_trial.params}")

In [ ]:
# ── Log best model to MLflow ──
mlflow.set_experiment("linear_models")
with mlflow.start_run():
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metric("best_cv_roc_auc", study.best_trial.value)
    mlflow.log_metric("n_trials", n_trials)

    # Train best model on full training set
    best_params = study.best_trial.params
    model_name = best_params.pop("model")
    if model_name == "lr":
        model = LogisticRegression(
            **best_params, solver="lbfgs", max_iter=1000, random_state=random_state
        )
    elif model_name == "svm":
        model = SVC(**best_params, probability=True, random_state=random_state)
    else:
        model = GaussianNB(**best_params)

    model.fit(X_train_scaled, y_train)
    mlflow.sklearn.log_model(model, "linear_model")
    print(f"Best linear model: {model_name}")
    print(f"Logged to MLflow run: {mlflow.active_run().info.run_id}")